# 3. Agentes con LangGraph (camino actual)

## Objetivos de Aprendizaje
- Conectar lo que ya viste en RA1 (`StateGraph`, `MessagesState`, `InMemorySaver`) con un **agente que usa herramientas**.
- Construir el ciclo ReAct como un **grafo visible**: nodo modelo → nodo tools → otra vez el modelo.
- Usar `create_agent`, la fábrica vigente de LangChain 1.x (por debajo es LangGraph).
- Diseñar un **grafo agentico** con ruta condicional (no todo es un loop ReAct).
- Armar criterio: cuándo grafo a mano, cuándo `create_agent`, cuándo el clásico `AgentExecutor`, cuándo CrewAI.

## Dónde estás en el módulo

1. **Fundamentos** — ReAct a mano, parsing de texto.
2. **Function calling** — el modelo devuelve JSON, no prosa.
3. **Este notebook** — el runtime que se usa hoy: un grafo.
4. **`3-langchain-agent.ipynb`** — el mismo problema con `AgentExecutor` (modo clásico; lo vas a ver en repos viejos).
5. **`4-crewai-agent.ipynb`** — equipos por roles, no un grafo.
6. **`5-criterio-frameworks.md`** — tabla para decidir con evidencia.

En RA1/IL1.1 ya compilaste un `StateGraph` de **un nodo** para memoria. Un agente es el mismo objeto con **un ciclo** y **herramientas**.


### 1. Instalación y configuración

In [ ]:
# Instalación de dependencias.
#
# Solo hace falta en Google Colab. En local, `uv sync` ya instaló todo esto con las
# versiones exactas del uv.lock; lanzar pip con -U aquí las actualizaría y rompería
# la reproducibilidad que el curso garantiza a todo el grupo.
import sys

if "google.colab" in sys.modules:
    %pip install -qU langchain-groq groq langgraph langchain requests python-dotenv
else:
    print("Entorno local: las dependencias ya las instaló uv sync.")


In [ ]:
import os
import re
import requests
from urllib.parse import quote

from langchain_groq import ChatGroq

try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

# Agente con tool calling nativo (create_agent / bind_tools).
# Por esta vía el modelo grande es el fiable; ver README de IL2.1.
MODELO = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

WIKIPEDIA_API_URL = "https://es.wikipedia.org/w/api.php"
WIKIPEDIA_SUMMARY_URL = "https://es.wikipedia.org/api/rest_v1/page/summary"
WIKIPEDIA_HEADERS = {
    "User-Agent": "Curso-IA-DUOC/1.0 (notebook educativo; contacto: estudiante@example.com)"
}

try:
    llm = ChatGroq(model=MODELO, temperature=0, reasoning_effort="low")
    print("✅ LLM configurado.")
    print(f"Modelo: {MODELO}")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None


### 2. La misma herramienta que el notebook clásico

El problema no cambia: preguntar por Marie Curie y consultar Wikipedia. Lo que cambia es **quién orquesta** el ciclo pensamiento → acción → observación.

`@tool` vive en `langchain_core`. Ya no hace falta importarlo desde `langchain_classic`.


In [ ]:
from langchain_core.tools import tool


def _wikipedia_get_json(url, params=None):
    try:
        response = requests.get(url, params=params, headers=WIKIPEDIA_HEADERS, timeout=10)
        response.raise_for_status()
        return response.json(), None
    except requests.exceptions.JSONDecodeError:
        return None, "Wikipedia devolvió una respuesta vacía o no válida en JSON."
    except requests.exceptions.RequestException as e:
        return None, f"No se pudo consultar Wikipedia: {e}"


def _limitar_oraciones(texto, max_oraciones=2):
    oraciones = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(oraciones[:max_oraciones])


@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Ideal para personas, lugares o conceptos."""
    if not query or not query.strip():
        return "No se recibió un término de búsqueda para Wikipedia."

    search_data, error = _wikipedia_get_json(
        WIKIPEDIA_API_URL,
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": 1,
            "format": "json",
            "utf8": 1,
        },
    )
    if error:
        return error

    results = search_data.get("query", {}).get("search", [])
    if not results:
        return f"No se encontró ninguna página para '{query}'."

    title = results[0]["title"]
    summary_data, error = _wikipedia_get_json(f"{WIKIPEDIA_SUMMARY_URL}/{quote(title)}")
    if error:
        return error

    extract = summary_data.get("extract")
    if not extract:
        return f"Wikipedia no entregó un resumen disponible para '{title}'."

    return _limitar_oraciones(extract, max_oraciones=2)


tools = [get_wikipedia_summary]
print("✅ Herramienta Wikipedia lista. El docstring ES lo que ve el modelo.")


### 3. El ciclo ReAct como grafo (a mano)

En el notebook 1 escribiste el `if/else` del loop. En el 2, Groq te devolvía `tool_calls`. Aquí el loop es el grafo:

```
START → modelo → (¿hay tool_calls?) → tools → modelo → …
                         ↓ no
                        END
```

`tools_condition` es esa pregunta. El nodo de herramientas **tiene que llamarse `tools`**: es el nombre que esa función espera.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

llm_con_tools = llm.bind_tools(tools)
SISTEMA = "Eres un asistente que busca en Wikipedia cuando necesita un dato. No inventes biografías."


def nodo_modelo(state: MessagesState):
    mensajes = [SystemMessage(content=SISTEMA), *state["messages"]]
    return {"messages": [llm_con_tools.invoke(mensajes)]}


grafo = StateGraph(MessagesState)
grafo.add_node("modelo", nodo_modelo)
grafo.add_node("tools", ToolNode(tools))  # el nombre 'tools' no es decorativo
grafo.add_edge(START, "modelo")
grafo.add_conditional_edges("modelo", tools_condition)
grafo.add_edge("tools", "modelo")
app_manual = grafo.compile()


def mostrar_grafo(app):
    """draw_ascii pide grandalf; este listado no añade dependencias."""
    g = app.get_graph()
    print("Nodos:", ", ".join(n.name for n in g.nodes.values()))
    for e in g.edges:
        marca = "  (condicional)" if getattr(e, "conditional", False) else ""
        print(f"  {e.source} → {e.target}{marca}")

print("✅ Grafo ReAct compilado a mano.")
mostrar_grafo(app_manual)


Invocamos con `messages`, no con `input`. El estado del grafo **es** la lista de mensajes: cada tool call y cada observación quedan ahí, visibles.


In [ ]:
consulta = "¿Quién fue Marie Curie y cuáles fueron sus logros más importantes?"
salida = app_manual.invoke({"messages": [HumanMessage(content=consulta)]})

print("=== Mensajes del grafo (así se depura un agente) ===")
for i, msg in enumerate(salida["messages"], 1):
    tipo = type(msg).__name__
    herramientas = getattr(msg, "tool_calls", None) or []
    extra = f"  tool_calls={ [c.get('name') for c in herramientas] }" if herramientas else ""
    texto = (getattr(msg, "content", "") or "")[:180].replace("\n", " ")
    print(f"{i}. {tipo}{extra}: {texto}")

print("\n🏁 Respuesta final:")
print(salida["messages"][-1].content)


### 4. `create_agent`: la fábrica actual

Hace el mismo grafo. No uses `langgraph.prebuilt.create_react_agent`: está **deprecado** a favor de `langchain.agents.create_agent`.

Tampoco uses `create_openai_tools_agent` + `AgentExecutor` para código nuevo. Eso queda en el notebook clásico, para que sepas leerlo.


In [ ]:
from langchain.agents import create_agent

agente = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SISTEMA,
)

print("✅ create_agent compiló un StateGraph.")
print("Tipo:", type(agente).__name__)
mostrar_grafo(agente)


In [ ]:
salida_fabrica = agente.invoke({"messages": [HumanMessage(content=consulta)]})
print(salida_fabrica["messages"][-1].content)


### 5. Grafo agentico con ruta (no todo es ReAct)

Un loop de herramientas es **una** topología. A veces quieres que el modelo **no** elija herramientas, sino que un nodo supervisor elija el camino: charla vs investigación.

Eso es un grafo agentico: estado tipado, arista condicional, nodos con un solo trabajo.

El nodo wiki **no** le pasa la pregunta cruda a Wikipedia: `¿Quién fue Marie Curie?` rankea *curia romana*. Un ReAct funciona porque el modelo extrae el nombre; un grafo con ruta tiene que hacer ese trabajo explícito.


In [ ]:
from typing import Literal, TypedDict


class EstadoRuta(TypedDict):
    consulta: str
    destino: str
    respuesta: str


def clasificar(state: EstadoRuta) -> dict:
    crudo = llm.invoke(
        [
            SystemMessage(content="Responde UNA palabra: wiki o charla. Sin explicación."),
            HumanMessage(content=state["consulta"]),
        ]
    ).content
    destino = "wiki" if "wiki" in (crudo or "").lower() else "charla"
    print(f"🧭 Ruta elegida: {destino} (el modelo dijo {crudo!r})")
    return {"destino": destino}


def termino_busqueda(texto: str) -> str:
    """Wikipedia busca títulos, no preguntas. '¿Quién fue Marie Curie?' cae en 'curia'."""
    limpio = re.sub(r"[¿?¡!]", "", texto)
    limpio = re.sub(
        r"(?i)\b(quién fue|quien fue|qué es|que es|háblame de|hablame de)\b",
        "",
        limpio,
    )
    return limpio.strip() or texto


def nodo_wiki(state: EstadoRuta) -> dict:
    termino = termino_busqueda(state["consulta"])
    dato = get_wikipedia_summary.invoke({"query": termino})
    print(f"🔎 Término: {termino!r}")
    print(f"📄 Wikipedia: {dato}")
    texto = llm.invoke(
        [
            SystemMessage(
                content=(
                    "Redacta 3-4 líneas con el extracto. "
                    "Si el extracto es un error, dilo en una frase."
                )
            ),
            HumanMessage(content=f"Pregunta: {state['consulta']}\nExtracto: {dato}"),
        ]
    ).content
    return {"respuesta": texto}


def nodo_charla(state: EstadoRuta) -> dict:
    texto = llm.invoke(
        [
            SystemMessage(content="Responde breve, sin buscar datos externos."),
            HumanMessage(content=state["consulta"]),
        ]
    ).content
    return {"respuesta": texto}


def enrutar(state: EstadoRuta) -> Literal["wiki", "charla"]:
    return "wiki" if state["destino"] == "wiki" else "charla"


rutas = StateGraph(EstadoRuta)
rutas.add_node("clasificar", clasificar)
rutas.add_node("wiki", nodo_wiki)
rutas.add_node("charla", nodo_charla)
rutas.add_edge(START, "clasificar")
rutas.add_conditional_edges("clasificar", enrutar)
rutas.add_edge("wiki", END)
rutas.add_edge("charla", END)
app_rutas = rutas.compile()

mostrar_grafo(app_rutas)


In [ ]:
for pregunta in (
    "¿Quién fue Marie Curie?",
    "Escríbeme un haiku sobre estudiar de noche",
):
    print(f"\n=== {pregunta} ===")
    out = app_rutas.invoke({"consulta": pregunta, "destino": "", "respuesta": ""})
    print(out["respuesta"])


### 6. Memoria: el checkpointer que ya conoces

En RA1 el `thread_id` recordaba un chat. Aquí recuerda un **agente con herramientas**. No hay `chat_history` que actualizar a mano.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agente_con_memoria = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SISTEMA,
    checkpointer=InMemorySaver(),
)
cfg = {"configurable": {"thread_id": "demo-marie"}}

r1 = agente_con_memoria.invoke(
    {"messages": [HumanMessage(content="Háblame de Marie Curie en 2 frases.")]},
    cfg,
)
print("Turno 1:", r1["messages"][-1].content, "\n")

r2 = agente_con_memoria.invoke(
    {"messages": [HumanMessage(content="¿En qué países vivió? No repitas la biografía.")]},
    cfg,
)
print("Turno 2 (sigue el hilo):", r2["messages"][-1].content)


## Criterio (llena `5-criterio-frameworks.md` después de correr el clásico y CrewAI)

| Pregunta | Grafo a mano | `create_agent` | `AgentExecutor` (clásico) | CrewAI |
|---|---|---|---|---|
| ¿Ves el ciclo? | Sí, nodos y aristas | Sí, pero ya armado | No: es una caja | No: ves roles y tasks |
| ¿Código nuevo en 2026? | Sí, si la topología no es ReAct | **Sí, default** | No | Sí, si hay equipo por roles |
| ¿Memoria? | `checkpointer` + `thread_id` | Igual | Lista `chat_history` | Depende del Crew |
| ¿Cuándo NO usarlo? | Si solo quieres un ReAct | Si necesitas una ruta rara | Si partes un repo nuevo | Si un agente basta |

**Regla:** si los "agentes" solo se pasan texto en línea recta, eso es una función con pasos, no un equipo. El grafo de la sección 5 es el medio: controlas la ruta sin inventar una empresa de roles.

Siguiente: `3-langchain-agent.ipynb` (el mismo Marie Curie, modo clásico) y `4-crewai-agent.ipynb` (investigador + escritor). Después cierra criterio en `5-criterio-frameworks.md`.
